In [1]:
# Cell 1: Import và cấu hình
import json
import sqlite3
import os
from pathlib import Path

# ========== CẤU HÌNH ==========
DB_PATH = "cookpad_clean.db"          # ← Đổi thành đường dẫn DB của bạn
AI_RESULTS_DIR = "ai_results_fixed"   # ← Folder chứa các file result_batch_X.json
SKIPPED_OUTPUT = "skipped_null_standard.json"  # File output cho các entry null
# ================================

print(f"DB: {DB_PATH}")
print(f"Results folder: {AI_RESULTS_DIR}")

DB: cookpad_clean.db
Results folder: ai_results_fixed


In [2]:
# Cell 2: Load tất cả file JSON từ folder ai_results_fixed

def load_all_results(folder_path: str) -> list[dict]:
    """Đọc toàn bộ file result_batch_X.json trong folder."""
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Không tìm thấy folder: {folder_path}")

    all_records = []
    files_loaded = []

    for file in sorted(folder.glob("*.json")):
        with open(file, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
                if isinstance(data, list) and len(data) > 0:
                    all_records.extend(data)
                    files_loaded.append(f"{file.name} ({len(data)} records)")
            except json.JSONDecodeError as e:
                print(f"⚠️  Lỗi đọc {file.name}: {e}")

    print(f"✅ Đã load {len(files_loaded)} files:")
    for f in files_loaded:
        print(f"   - {f}")
    print(f"📦 Tổng: {len(all_records)} records")
    return all_records


all_records = load_all_results(AI_RESULTS_DIR)

✅ Đã load 626 files:
   - result_batch_0.json (28 records)
   - result_batch_1.json (29 records)
   - result_batch_10.json (28 records)
   - result_batch_100.json (29 records)
   - result_batch_101.json (28 records)
   - result_batch_102.json (30 records)
   - result_batch_103.json (29 records)
   - result_batch_104.json (29 records)
   - result_batch_105.json (29 records)
   - result_batch_106.json (29 records)
   - result_batch_107.json (30 records)
   - result_batch_108.json (30 records)
   - result_batch_109.json (27 records)
   - result_batch_11.json (30 records)
   - result_batch_110.json (27 records)
   - result_batch_111.json (26 records)
   - result_batch_112.json (30 records)
   - result_batch_113.json (28 records)
   - result_batch_114.json (29 records)
   - result_batch_115.json (30 records)
   - result_batch_116.json (28 records)
   - result_batch_117.json (29 records)
   - result_batch_118.json (28 records)
   - result_batch_119.json (29 records)
   - result_batch_12.json

In [3]:
# Cell 3: Tách valid records vs null records

def split_records(records: list[dict]):
    """
    Tách thành 2 nhóm:
    - valid: có standard_id → sẽ insert vào DB
    - null_records: standard_id là None → ghi ra file riêng
    """
    valid = []
    null_records = []

    for r in records:
        if r.get("standard_id") is not None:
            valid.append(r)
        else:
            null_records.append({
                "recipe_id": r.get("recipe_id"),
                "raw_name": r.get("raw_name")
            })

    print(f"✅ Valid (có standard_id): {len(valid)}")
    print(f"⚠️  Null standard_id: {len(null_records)}")
    return valid, null_records


valid_records, null_records = split_records(all_records)

✅ Valid (có standard_id): 18096
⚠️  Null standard_id: 58


In [4]:
# Cell 5: Kiểm tra duplicate và insert vào DB

def update_dish_ingredient(db_path: str, valid_records: list[dict]):
    """
    Với mỗi record có standard_id:
    - Nếu (recipe_id, ingredient_id) đã tồn tại → bỏ qua
    - Nếu chưa → INSERT (quantity_g, is_main, is_optional để trống/NULL)
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    inserted = 0
    skipped_dup = 0
    errors = []

    CHECK_SQL = """
        SELECT COUNT(*) FROM dish_ingredient
        WHERE recipe_id = ? AND ingredient_id = ?
    """

    INSERT_SQL = """
        INSERT INTO dish_ingredient (recipe_id, ingredient_id)
        VALUES (?, ?)
    """

    for r in valid_records:
        recipe_id = r["recipe_id"]
        ingredient_id = r["standard_id"]

        try:
            # Kiểm tra đã tồn tại chưa
            cursor.execute(CHECK_SQL, (recipe_id, ingredient_id))
            count = cursor.fetchone()[0]

            if count > 0:
                skipped_dup += 1
            else:
                cursor.execute(INSERT_SQL, (recipe_id, ingredient_id))
                inserted += 1

        except sqlite3.Error as e:
            errors.append({
                "recipe_id": recipe_id,
                "ingredient_id": ingredient_id,
                "error": str(e)
            })

    conn.commit()
    conn.close()

    print(f"✅ Inserted:      {inserted}")
    print(f"⏭️  Skipped (dup): {skipped_dup}")
    print(f"❌ Errors:        {len(errors)}")

    if errors:
        print("\nChi tiết lỗi:")
        for e in errors[:10]:
            print(f"  recipe={e['recipe_id']}, ing={e['ingredient_id']} → {e['error']}")

    return inserted, skipped_dup, errors


inserted, skipped, errors = update_dish_ingredient(DB_PATH, valid_records)

✅ Inserted:      10932
⏭️  Skipped (dup): 7164
❌ Errors:        0


In [5]:
# Cell 6: Verify kết quả

def verify_results(db_path: str, sample_records: list[dict], n=5):
    """Kiểm tra ngẫu nhiên một số record đã được insert."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    print(f"🔍 Kiểm tra {n} records ngẫu nhiên:\n")
    import random
    samples = random.sample(sample_records, min(n, len(sample_records)))

    for r in samples:
        cursor.execute(
            "SELECT id, recipe_id, ingredient_id FROM dish_ingredient WHERE recipe_id=? AND ingredient_id=?",
            (r["recipe_id"], r["standard_id"])
        )
        row = cursor.fetchone()
        status = "✅ EXISTS" if row else "❌ NOT FOUND"
        print(f"  {status} | recipe={r['recipe_id']}, ing_id={r['standard_id']} ({r.get('standard_name','')})")

    conn.close()

    # Tổng count
    conn = sqlite3.connect(db_path)
    total = conn.execute("SELECT COUNT(*) FROM dish_ingredient").fetchone()[0]
    conn.close()
    print(f"\n📊 Tổng rows trong dish_ingredient: {total}")


verify_results(DB_PATH, valid_records)

🔍 Kiểm tra 5 records ngẫu nhiên:

  ✅ EXISTS | recipe=16484783, ing_id=7515 (bột asafoetida)
  ✅ EXISTS | recipe=24857985, ing_id=151 (nấm)
  ✅ EXISTS | recipe=24405443, ing_id=3638 (rau nêm)
  ✅ EXISTS | recipe=15502724, ing_id=11829 (quế)
  ✅ EXISTS | recipe=17021371, ing_id=360 (đậu xanh)

📊 Tổng rows trong dish_ingredient: 43109


In [6]:
import json
from pathlib import Path
from collections import defaultdict

AI_RESULTS_DIR = "ai_results_fixed"
SKIPPED_OUTPUT = "skipped_null_ingredients.json"

# Load tất cả records
all_records = []
for file in sorted(Path(AI_RESULTS_DIR).glob("*.json")):
    with open(file, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            if isinstance(data, list):
                all_records.extend(data)
        except json.JSONDecodeError as e:
            print(f"⚠️ Lỗi đọc {file.name}: {e}")

# Nhóm theo recipe_id, tìm recipe có ít nhất 1 null
grouped = defaultdict(list)
for r in all_records:
    grouped[r["recipe_id"]].append(r)

skipped = []
for recipe_id, ingredients in grouped.items():
    null_ings = [i for i in ingredients if i.get("standard_id") is None]
    if null_ings:
        skipped.append({
            "recipe_id": recipe_id,
            "null_ingredients": [{"raw_id": i.get("raw_id"), "raw_name": i.get("raw_name")} for i in null_ings]
        })

# Ghi ra file
with open(SKIPPED_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print(f"✅ Tìm thấy {len(skipped)} recipes có null ingredient → {SKIPPED_OUTPUT}")

✅ Tìm thấy 58 recipes có null ingredient → skipped_null_ingredients.json
